# FaceSwap Demo — Google Colab

**GitHub × Drive 構成:**
- ソースコード → GitHub から毎回最新を取得
- モデル・フレームキャッシュ → Drive に保存して使いまわし

**起動手順:**
1. **Cell 1（設定セル）の `NGROK_TOKEN` を書き換える**（必須）
2. `Runtime` → `Run all` で全セル実行
3. Cell 6 に表示された URL を開く

**2回目以降は `NGROK_TOKEN` を確認して `Run all` するだけ。**

In [ ]:
# ============================================================
# ⚙️  設定 — ここだけ書き換えてください
# ============================================================

# ngrok authtoken（必須）→ https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = 'YOUR_NGROK_TOKEN'

# GitHub リポジトリ URL（変更不要）
REPO_URL = 'https://github.com/oyatuchokobi/faceswap.git'

# Google Drive 上の保存先（変更不要）
DRIVE_DIR = '/content/drive/MyDrive/faceswap-data'

# ============================================================

In [ ]:
# Cell 2: Drive マウント & GitHub からコード取得
import os, subprocess, shutil
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

REPO_DIR = '/content/faceswap'

# --- GitHub クローン / プル ---
if Path(f'{REPO_DIR}/.git').exists():
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--rebase'], check=True)
    print('✅ GitHub から最新コードを取得しました')
else:
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    print('✅ GitHub からクローンしました')

# --- Drive の保存ディレクトリを準備 ---
for d in ['models', 'templates/target']:
    Path(f'{DRIVE_DIR}/{d}').mkdir(parents=True, exist_ok=True)

# --- シンボリックリンク: models/ → Drive ---
models_link = Path(f'{REPO_DIR}/models')
if models_link.exists() and not models_link.is_symlink():
    shutil.rmtree(str(models_link))
if not models_link.is_symlink():
    models_link.symlink_to(f'{DRIVE_DIR}/models')

# --- シンボリックリンク: templates/target/ → Drive (フレームキャッシュ) ---
frames_link = Path(f'{REPO_DIR}/templates/target')
frames_link.parent.mkdir(parents=True, exist_ok=True)
if frames_link.exists() and not frames_link.is_symlink():
    shutil.rmtree(str(frames_link))
if not frames_link.is_symlink():
    frames_link.symlink_to(f'{DRIVE_DIR}/templates/target')

print(f'✅ Drive 連携完了')
print(f'   models/      → {DRIVE_DIR}/models')
print(f'   template frames → {DRIVE_DIR}/templates/target')

# --- モデルの有無を事前確認 ---
model_path = Path(f'{DRIVE_DIR}/models/inswapper_128.onnx')
if model_path.exists():
    print('✅ モデル確認済み（Cell 4 のダウンロードはスキップされます）')
else:
    print('⚠️  モデルが見つかりません → Cell 4 でダウンロードします（約 554MB、数分かかります）')

In [ ]:
# Cell 3: 依存パッケージのインストール
import subprocess, sys

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'fastapi==0.115.0',
    'uvicorn[standard]==0.32.0',
    'python-multipart==0.0.12',
    'insightface==0.7.3',
    'onnxruntime-gpu==1.20.0',
    'numpy>=1.24,<2.0',
    'opencv-python-headless==4.10.0.84',
    'Pillow==11.0.0',
    'aiofiles==24.1.0',
    'sse-starlette==2.1.3',
    'pyngrok',
], check=True)
print('✅ インストール完了')

import onnxruntime as ort
providers = ort.get_available_providers()
if 'CUDAExecutionProvider' in providers:
    print(f'✅ GPU推論が有効 (CUDA)')
else:
    print(f'⚠️  CUDAなし → CPU推論になります。プロバイダー: {providers}')

In [ ]:
# Cell 4: モデル確認 / ダウンロード（初回のみ Drive に保存）
import subprocess, sys
from pathlib import Path

model = Path(f'{REPO_DIR}/models/inswapper_128.onnx')

if not model.exists():
    print('モデルをダウンロード中... (初回のみ、約 554MB)')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'], check=True)
    from huggingface_hub import hf_hub_download
    hf_hub_download(
        repo_id='ezioruan/inswapper_128.onnx',
        filename='inswapper_128.onnx',
        local_dir=str(model.parent),
    )
    print('✅ ダウンロード完了・Drive に保存しました（次回以降は不要）')
else:
    print(f'✅ モデル確認済み（Drive より使用）')

In [ ]:
# Cell 5: uvicorn をバックグラウンドで起動
import subprocess, sys, time, urllib.request, urllib.error

uvicorn_proc = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'backend.main:app',
     '--host', '0.0.0.0', '--port', '8000'],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

# ヘルスチェックで起動確認（最大 30 秒）
for _ in range(30):
    time.sleep(1)
    if uvicorn_proc.poll() is not None:
        out = uvicorn_proc.stdout.read().decode()
        raise RuntimeError(f'uvicorn の起動に失敗しました:\n{out}')
    try:
        urllib.request.urlopen('http://localhost:8000/api/health', timeout=1)
        print('✅ uvicorn 起動済み (http://localhost:8000)')
        break
    except urllib.error.URLError:
        pass
else:
    raise RuntimeError('uvicorn が 30 秒以内に起動しませんでした')


In [ ]:
# Cell 6: ngrok トンネル → アプリ URL を表示
from pyngrok import ngrok, conf

conf.get_default().auth_token = NGROK_TOKEN
tunnel = ngrok.connect(8000)

print(f'\n🎉 アプリURL: {tunnel.public_url}')
print('このURLをブラウザで開いてください（スマホも可）')
print(f'デバッグ用: {tunnel.public_url}/?mode=stg')
